# L.I.V.E. — treino do modelo acústico (Colab, GPU)

Tudo sintético ou público (Princípio IV): voz Kokoro (Apache-2.0) + ruído MUSAN (CC BY 4.0). A gravação real do PO **nunca** vem para cá.

Pipeline: ambiente Python 3.12 → falas (mesma semente = mesmas falas da máquina local) → MUSAN → preparação (os arquivos reservados para a gravação ficam de fora, pela mesma regra) → treino retomável.

In [ ]:
BRANCH = "002-modelo-ctc-onnx"
SEED, COUNT, COPIES, EPOCHS = 7, 3000, 4, 80
!git clone -q --depth 1 -b {BRANCH} https://github.com/yuriwinchest/live-engine-training.git
%cd /content/live-engine-training/training
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q uv && uv venv -q -p 3.12 /content/venv312
V = "/content/venv312/bin/python"
# --python explícito: o Colab define UV_SYSTEM_PYTHON=true.
!uv pip install -q --python {V} -e '.[train]' 'kokoro>=0.9.4' 'transformers>=4.40'
!{V} -c "import torch; print('GPU:', torch.cuda.is_available())"

## Falas sintéticas (≈ 7 min na T4)

In [ ]:
L = f"{V} -m live_lab"
!{L} synth plan --engine kokoro --out data/jobs-kokoro.jsonl --count {COUNT} --seed {SEED}
!{L} synth render --engine kokoro --jobs data/jobs-kokoro.jsonl --out data/clean/kokoro

## Ruído MUSAN (OpenSLR 17, CC BY 4.0) — só `noise` e `music`

In [ ]:
!test -d /content/musan || (wget -q -O /content/musan.tar.gz https://openslr.trmal.net/resources/17/musan.tar.gz && tar -xzf /content/musan.tar.gz -C /content musan/noise musan/music && rm /content/musan.tar.gz)
ATTR = "MUSAN: A Music, Speech, and Noise Corpus — D. Snyder, G. Chen, D. Povey (2015), arXiv:1510.08484"
fontes = f"""
[[source]]
name = "kokoro-82m"
kind = "speech_synthetic"
license = "Apache-2.0"
origin = "https://github.com/hexgrad/kokoro"
obtained_at = 2026-10-03

[[source]]
name = "musan-noise"
kind = "noise"
license = "CC-BY-4.0"
origin = "https://www.openslr.org/17/"
obtained_at = 2026-10-03
path = "/content/musan/noise"
attribution = "{ATTR}"

[[source]]
name = "musan-music"
kind = "noise"
license = "CC-BY-4.0"
origin = "https://www.openslr.org/17/"
obtained_at = 2026-10-03
path = "/content/musan/music"
attribution = "{ATTR}"
"""
open("data/fontes.toml", "w", encoding="utf-8").write(fontes)
!{L} sources check --registry data/fontes.toml

## Preparação (degradação com ruído, partições por voz)

In [ ]:
!{L} prep --registry data/fontes.toml --clean data/clean/kokoro/clean.jsonl --out data/ds-v1 --seed {SEED} --copies {COPIES}
!cat data/ds-v1/report.md

## Treino

Os pontos de retomada ficam em `data/run-v1`. Se a célula parar no meio, rode de novo: continua da última época salva (dentro da mesma sessão do Colab).

In [ ]:
!{L} train --manifest data/ds-v1/manifest.jsonl --out data/run-v1 --seed {SEED} --epochs {EPOCHS}

## Levar o melhor modelo para a máquina local (exportação e avaliação rodam lá)

In [ ]:
!cd data && zip -q /content/run-v1.zip run-v1/best.pt run-v1/config.json run-v1/history.json
from google.colab import files
files.download("/content/run-v1.zip")